##### Feladat (Példatár 3.1)

Egy $a \times a$ állandó négyzet keresztmetszetű, rugalmas betonhasábot merev fal vesz körül. A hasábot egy merev fedélen keresztül az $F$ nyomóerővel terheljük. Mekkora feszültségek ébrednek a hasáb belső pontjaiban és mekkora lesz a $h$ magasságú hasáb zsugorodása?

Adatok: $a = 200 ~\mathrm{mm}$, $F = 160 ~\mathrm{kN}$, $E = 50 ~\mathrm{GPa}$, $\nu = 0.35$, $h = 1 ~\mathrm{m}$.

Feladatok:

a) Feszültségek a hasáb belső pontjaiban?

b) Mekkora a $h$ magasságú hasáb zsugorodása?

<img src="img/01.jpg" alt="Feladat" width=300px>

##### Csomagok importálása

In [1]:
import numpy as np # Numerikus számítások
import matplotlib.pyplot as plt # Ábrázolás
plt.rcParams['mathtext.fontset'] = 'cm' # Betűtípus az ábrákon
plt.rcParams['font.family'] = 'serif' # Betűtípus az ábrákon
plt.rcParams['font.serif'] = ['cmr10'] # Betűtípus az ábrákon
plt.rcParams['axes.formatter.use_mathtext'] = True # Betűtípus az ábrákon
plt.rcParams['axes.unicode_minus'] = False # Betűtípus az ábrákon
from IPython.display import display, Math, Latex # Szép kimenet Jupyter notebookban
import sympy as sp # Szimbólikus számítások
from sympy import latex # Szép kimenet Jupyter notebookban
sp.init_printing() # Szép kimenet Jupyter notebookban

##### Adatok

In [2]:
a_data = 200 # mm
F_data = 160 # kN
E_data = 50 # GPa
nu_data = 0.35
h_data = 1 # m

# Átváltás N, mm, MPa rendszerbe
F_data = F_data * 10**3 # N
E_data = E_data * 10**3 # MPa
h_data = h_data * 10**3 # mm

In [3]:
a, F, E, nu, h = sp.symbols('a, F, E, nu, h')

In [4]:
data = [(a, a_data),
        (F, F_data),
        (E, E_data),
        (nu, nu_data),
        (h, h_data)]

#### a) Feszültségek a hasáb belső pontjaiban

##### Alakváltozási állapot

A merev falak miatt a hasáb $x$ és $y$ irányban nem tud alakváltozni, így az alakváltozási állapot:

In [5]:
eps_z = sp.Symbol('varepsilon_z') # Ismeretlen fajlagos nyúlás z irányban

# Alakváltozási tenzor
eps = sp.Matrix([[0, 0,     0],
                 [0, 0,     0],
                 [0, 0, eps_z]])
display(Math(rf'\boldsymbol{{\varepsilon}} = {latex(eps)}, \qquad \text{{ahol}} \quad \varepsilon_z = \frac{{\Delta h}}{{h}}'))

<IPython.core.display.Math object>

##### Feszültségi állapot

In [6]:
sigma_x, sigma_y, sigma_z = sp.symbols('sigma_x, sigma_y, sigma_z') # Ismeretlen normálfeszültségek

# Feszültségi tenzor
sigma = sp.Matrix([[sigma_x,       0,       0],
                   [      0, sigma_y,       0],
                   [      0,       0, sigma_z]])
display(Math(rf'\boldsymbol{{\sigma}} = {latex(sigma)}'))

<IPython.core.display.Math object>

##### Általános Hooke törvény

$$\boldsymbol{\sigma} = \frac{E}{1+\nu} \left( \boldsymbol{\varepsilon} + \frac{\nu}{1-2\nu} \cdot \varepsilon_I \cdot \mathbf{E} \right)$$

ahol:
- $\boldsymbol{\sigma}$: a feszültségi tenzor,
- $\boldsymbol{\varepsilon}$: az alakváltozási tenzor,
- $E$: a rugalmassági modulusz,
- $\nu$: a Poisson-tényező,
- $\varepsilon_I = \mathrm{tr}(\boldsymbol{\varepsilon})$: az alakváltozási tenzor első skalárinvariánsa (a főátlóbeli elemek összege),
- $\mathbf{E}$: az egységmátrix.

<img src="img/02.jpg" alt="Feladat" width=600px>

In [7]:
I = sp.eye(3) # Egységmátrix

# Az alakváltozási tenzor első skalárinvariánsa
eps_I = eps.trace()
display(Math(rf'\varepsilon_I = \mathrm{{tr}}(\boldsymbol{{\varepsilon}}) = {latex(eps_I)}'))

<IPython.core.display.Math object>

##### Behelyettesítve

In [8]:
sigma_Hooke = E / (1 + nu) * (eps + nu / (1 - 2*nu) * eps_I * I)

# Közös nevezőre hozás
sigma_Hooke = sigma_Hooke.applyfunc(sp.together)

display(Math(rf'{latex(sigma)} = {latex(sigma_Hooke)}'))

<IPython.core.display.Math object>

In [9]:
# Főátló 1. és 2. eleme (x és y irány)
display(Math(rf'\sigma_x = \sigma_y = {latex(sigma_Hooke[0, 0])} \qquad (1)'))

# Főátló 3. eleme (z irány)
display(Math(rf'\sigma_z = {latex(sigma_Hooke[2, 2])} \qquad (2)'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

##### $\sigma_z$ ismert a terhelésből

In [10]:
sigma_z_sol = -F / a**2
display(Math(rf'\sigma_z = -\frac{{F}}{{a^2}} = {latex(sigma_z_sol.subs(data))} ~\mathrm{{MPa}}'))

<IPython.core.display.Math object>

##### (2)-ből a fajlagos nyúlás

In [11]:
eq = sigma_Hooke[2, 2] - sigma_z_sol

# Rendszer megoldása
sol = sp.solve(eq, eps_z)
eps_z_sol = sol[0]
display(Math(rf'\varepsilon_z = \frac{{(1+\nu)(1-2\nu)}}{{E (1-\nu)}} \cdot \sigma_z = {latex(eps_z_sol.subs(data).evalf(4))}'))

<IPython.core.display.Math object>

##### Vissza (1)-be

In [12]:
sigma_x_sol = sigma_Hooke[0, 0].subs(eps_z, eps_z_sol)
display(Math(rf'\sigma_x = \sigma_y = {latex(sigma_x_sol.subs(data).evalf(4))} ~\mathrm{{MPa}}'))

<IPython.core.display.Math object>

##### Így az alakváltozási és a feszültségi állapot

In [13]:
eps_sol = eps.subs(eps_z, eps_z_sol).subs(data)
sigma_sol = sigma_Hooke.subs(eps_z, eps_z_sol).subs(data)

display(Math(rf'\boldsymbol{{\varepsilon}} = {latex((eps_sol * 10**5).evalf(4))} \cdot 10^{{-5}}'))
display(Math(rf'\boldsymbol{{\sigma}} = {latex(sigma_sol.evalf(4))} ~\mathrm{{MPa}}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

#### b) A $h$ magasságú hasáb zsugorodása

In [14]:
# ε_z = Δh / h
delta_h = h * eps_z_sol
display(Math(rf'\Delta h = h \cdot \varepsilon_z = {latex(delta_h.subs(data).evalf(4))} ~\mathrm{{mm}}'))

<IPython.core.display.Math object>

A negatív előjel azt jelenti, hogy a hasáb magassága csökken, vagyis a hasáb zsugorodik.